# 🛡️ TrueVoice — Deepfake Voice Detection Demo

**Gemma 4 E4B audio_tower** based deepfake voice detection.
Trained on ASVspoof 2019 LA | EER 5.20% on ASVspoof 2021 LA eval (148,176 samples)

---

## Before You Start
- ✅ Select a **GPU runtime** (Runtime → Change runtime type → T4 or higher)
- ✅ Set your HuggingFace token as a Colab secret named `HF_TOKEN`
  - You need access to `google/gemma-4-e4b-it` on HuggingFace
  - Colab sidebar → 🔑 Secrets → Add `HF_TOKEN`

## Execution Order
| Cell | Description |
|------|-------------|
| 1 | Install packages |
| 2 | Download model weights from GitHub |
| 3 | Define preprocessing functions |
| 4 | Load Gemma 4 E4B model & processor |
| 5 | Define AudioDeepfakeClassifier |
| 6 | Restore classifier head weights |
| 7 | Launch Gradio demo |

## Cell 1: Install Packages

In [ ]:
%%capture
!pip install git+https://github.com/huggingface/transformers.git
!pip install librosa soundfile torchaudio gradio accelerate scikit-learn scipy
print('Packages installed successfully')

## Cell 2: Download Model Weights from GitHub

Downloads `classifier_head.pt` directly from the repository. No Google Drive required.

In [ ]:
import os

REPO     = 'https://raw.githubusercontent.com/hyeminss11/true-voice-gemma4/main'
SAVE_DIR = '/content/saved_model'
DEVICE   = 'cuda'
os.makedirs(SAVE_DIR, exist_ok=True)

!wget -q -O {SAVE_DIR}/classifier_head.pt {REPO}/saved_model/classifier_head.pt
!wget -q -O {SAVE_DIR}/metadata.json       {REPO}/saved_model/metadata.json

pt_size = os.path.getsize(f'{SAVE_DIR}/classifier_head.pt') / 1e6
print(f'classifier_head.pt : {pt_size:.1f} MB')
print(f'metadata.json      : downloaded')

if pt_size < 1.0:
    raise RuntimeError('Download failed. Check that the repository is public and files exist.')

print('Download complete')

## Cell 3: Define Preprocessing Functions

In [ ]:
import torch
import torchaudio
import librosa


def codec_simulate(audio_np, sr=16000):
    """Simulate phone codec: downsample to 8kHz then upsample back to 16kHz.
    Used in Phone Call Simulation Mode in the Gradio UI.
    """
    waveform = torch.from_numpy(audio_np).unsqueeze(0).float()
    down = torchaudio.functional.resample(waveform, orig_freq=sr, new_freq=8000)
    up   = torchaudio.functional.resample(down, orig_freq=8000, new_freq=sr)
    return up.squeeze(0).numpy()


def load_audio(path, sr=16000, max_sec=29.0, apply_codec=False):
    """Load audio, clip to max_sec, and optionally apply codec simulation."""
    audio, _ = librosa.load(path, sr=sr, mono=True)
    max_samples = int(max_sec * sr)
    if len(audio) > max_samples:
        audio = audio[:max_samples]
    if apply_codec:
        audio = codec_simulate(audio, sr=sr)
    return audio


print('Preprocessing functions defined')

## Cell 4: Load Model & Processor

> Only the `audio_tower` is used — the language model backbone is fully frozen.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from google.colab import userdata
import torch

HF_TOKEN = userdata.get('HF_TOKEN')
MODEL_ID = 'google/gemma-4-e4b-it'

print(f'Device     : {DEVICE}')
print(f'GPU memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
print('Processor loaded')

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    dtype=torch.float32,
    device_map='auto',
)
print('Base model loaded')

audio_tower = base_model.model.audio_tower
hidden_size = audio_tower.output_proj.out_features  # 1536
print(f'audio_tower hidden_size: {hidden_size}')

## Cell 5: Define AudioDeepfakeClassifier

In [ ]:
import torch.nn as nn
from transformers.modeling_outputs import SequenceClassifierOutput


class AudioDeepfakeClassifier(nn.Module):
    """Binary classifier using Gemma 4 E4B's audio_tower as a frozen feature extractor.

    Architecture:
        audio_tower (frozen, 1536-dim) -> mean pooling
        -> Linear(1536->256) -> GELU -> Dropout(0.3) -> Linear(256->2)
    """
    def __init__(self, audio_tower, hidden_size, num_labels=2, dropout_p=0.3):
        super().__init__()
        self.audio_tower = audio_tower
        self.num_labels  = num_labels
        self.classifier  = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_features, labels=None):
        tower_out = self.audio_tower(input_features=input_features)
        hidden    = tower_out.last_hidden_state if hasattr(tower_out, 'last_hidden_state') else tower_out[0]
        pooled    = hidden.mean(dim=1)
        logits    = self.classifier(pooled)
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return SequenceClassifierOutput(loss=loss, logits=logits)


print('AudioDeepfakeClassifier defined')

## Cell 6: Restore Classifier Head Weights

In [ ]:
import torch

# Freeze audio_tower
for param in base_model.model.audio_tower.parameters():
    param.requires_grad = False

# Build classifier (audio_tower already on GPU via device_map='auto')
clf_model = AudioDeepfakeClassifier(
    audio_tower=base_model.model.audio_tower,
    hidden_size=hidden_size,
)

# Move classifier head to GPU (float32 to match base model)
clf_model.classifier = clf_model.classifier.to(DEVICE).to(torch.float32)

# Load classifier head weights from downloaded file
clf_model.classifier.load_state_dict(
    torch.load(f'{SAVE_DIR}/classifier_head.pt', map_location='cpu')
)

print('Model restored successfully')

## Cell 7: Launch Gradio Demo

Running this cell generates a **public URL valid for 72 hours**.

In [ ]:
import gradio as gr
import numpy as np
import torch
import librosa


def predict_deepfake(audio_input, phone_mode):
    """Gradio callback: receives audio input and returns real/fake prediction."""

    if audio_input is None:
        return '⚠️ Please upload an audio file.', {}, '', '<div style="text-align:center; color:gray;">Waiting for input...</div>'

    if isinstance(audio_input, tuple):
        sr, audio_array = audio_input
        audio = audio_array.astype(np.float32)
        if audio.max() > 1.0:
            audio = audio / 32768.0
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != 16000:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    else:
        audio, _ = librosa.load(audio_input, sr=16000, mono=True)

    if np.abs(audio).max() < 0.005:
        return '⚠️ No voice detected.', {}, '', '<div style="text-align:center; color:gray;">Silent audio detected</div>'

    if len(audio) < 16000 * 1.5:
        return '⚠️ Audio too short (minimum 1.5 seconds required).', {}, '', '<div style="text-align:center; color:gray;">Clip too short</div>'

    if phone_mode:
        audio = codec_simulate(audio, sr=16000)

    audio    = audio[:int(29.0 * 16000)]
    duration = len(audio) / 16000

    inputs = processor(
        text='<audio>',
        audio=audio,
        sampling_rate=16000,
        return_tensors='pt',
    )

    clf_model.eval()
    with torch.no_grad():
        out   = clf_model(input_features=inputs['input_features'].to(DEVICE))
        probs = torch.softmax(out.logits, dim=-1)[0].cpu().numpy()

    real_prob = float(probs[0])
    fake_prob = float(probs[1])
    fake_pct  = fake_prob * 100

    if fake_pct >= 70:
        label     = '🔴 Deepfake Detected'
        risk      = '⚠️ Risk Level: HIGH'
        bar_color = '#e53935'
    elif fake_pct >= 40:
        label     = '🟡 Possible Deepfake'
        risk      = '⚠️ Risk Level: MEDIUM'
        bar_color = '#fb8c00'
    else:
        label     = '🟢 Real Voice'
        risk      = '✅ Risk Level: LOW'
        bar_color = '#43a047'

    gauge_html = f"""
    <div style="padding: 12px;">
        <div style="display:flex; justify-content:space-between; margin-bottom:6px;">
            <span style="font-weight:bold; font-size:15px;">Deepfake Risk Score</span>
            <span style="font-weight:bold; font-size:15px; color:{bar_color};">{fake_pct:.1f}%</span>
        </div>
        <div style="background:#e0e0e0; border-radius:10px; height:24px; overflow:hidden;">
            <div style="width:{fake_pct}%; background:{bar_color}; height:100%; border-radius:10px;"></div>
        </div>
        <div style="display:flex; justify-content:space-between; margin-top:4px; font-size:12px; color:#888;">
            <span>0% Safe</span>
            <span>40% Caution</span>
            <span>70% Danger</span>
        </div>
    </div>
    """

    mode_info = '📞 Phone Call Mode applied' if phone_mode else '🎙️ Standard Mode'
    info = f'{risk}  |  {mode_info}  |  Duration: {duration:.1f}s'
    conf = {'Real Voice': real_prob, 'Deepfake': fake_prob}

    return label, conf, info, gauge_html


with gr.Blocks(title='🛡️ TrueVoice') as demo:
    gr.Markdown("""
    # 🛡️ TrueVoice — Deepfake Voice Detection
    **Gemma 4 E4B audio_tower** based real-time voice authenticity detection
    > Trained on ASVspoof 2019 | EER 5.20% (2021 LA eval, 148K samples)
    """)

    with gr.Row():
        with gr.Column():
            audio_input = gr.Audio(
                label='🎙️ Upload Audio File (max 29 seconds)',
                type='numpy',
            )
            phone_mode = gr.Checkbox(
                label='📞 Phone Call Mode (8kHz codec simulation)',
                value=False,
            )
            submit_btn = gr.Button('🔍 Analyze', variant='primary')

        with gr.Column():
            label_out = gr.Textbox(label='Verdict', interactive=False)
            gauge_out = gr.HTML(label='Risk Gauge')
            prob_out  = gr.Label(label='Probability', num_top_classes=2)
            info_out  = gr.Textbox(label='Details', interactive=False)

    submit_btn.click(
        fn=predict_deepfake,
        inputs=[audio_input, phone_mode],
        outputs=[label_out, prob_out, info_out, gauge_out],
    )

    gr.Markdown("""
    ### How to Use
    1. Upload an audio file or record via microphone
    2. Enable 📞 Phone Call Mode if analyzing a phone call recording
    3. Click Analyze
    """)

demo.launch(share=True)